# Module 2: Web search with Tavily - your first agentic loop

Large language models are frozen in time — they only know what was in their training data. To answer questions about *what's happening now* (say, this week's AI news), we have to **give the model a tool** and let it go fetch fresh information.

In this notebook we build that from scratch, with no agent framework:

1. Call the **Tavily** search API directly over HTTP, then via its Python client.
2. Turn search into a **tool** — a plain Python function the model can call.
3. Wire a **ReAct-style agentic loop** in ~20 lines: the model reasons, calls the tool, reads the results, and repeats until it can answer.

By the end you'll have a tiny research agent that answers a question with real, cited sources — and you'll understand every line of it.

## 0 — Get your API keys

You need two keys, both placed in a `.env` file inside this `lesson/` folder. Copy over the `.env.example` to a new file, rename it to `.env`.

**Tavily** (web search)
1. Go to [tavily.com](https://www.tavily.com) and sign in with Google or GitHub.
2. Copy an API key from the dashboard — it looks like `tvly-...`.
3. Free tier: **1,000 credits/month, no credit card**. A basic search costs 1 credit.

**Nebius Token Factory** (the LLM)
1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **API keys → Create API key** and copy it (you can't view it again later).
3. Nebius exposes an **OpenAI-compatible** API, so we drive it with the plain `openai` SDK.

Your `lesson/.env` should look like:

```
TAVILY_API_KEY=tvly-...
NEBIUS_API_KEY=...
```

In [ ]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv()
%load_ext rich

assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY in lesson/.env"
assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

## 1 — Call Tavily directly over HTTP

Before any client library or agent framework, we will see what Tavily Search *actually is*. At its core, it's a single HTTP endpoint you POST a query to. Everything else is a wrapper around this.

- **Endpoint:** `POST https://api.tavily.com/search`
- **Auth:** an `Authorization: Bearer <your-key>` header
- **Body:** JSON with your `query` plus a few optional knobs

The parameters worth knowing:

| Param | What it does |
|-------|--------------|
| `query` | The search string (required) |
| `search_depth` | `basic`/`fast` (1 credit) or `advanced` (2 credits, deeper) |
| `max_results` | How many results to return (1–20) |
| `chunks_per_source` | How many snippets to return per source (1–3) |
| `include_answer` | Also return a one-shot LLM answer to the query |
| `time_range` | `day` / `week` / `month` / `year` to filter by recency |
| `include_raw_content` | Return the raw HTML content of the page |

In [ ]:
import requests

response = requests.post(
    "https://api.tavily.com/search",
    headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
    json={
        "query": "advancements in AI this week",
        "search_depth": "advanced",
        "max_results": 5,
        "time_range": "week",
        "include_answer": True,
    },
    timeout=30,
)
response.raise_for_status()
data = response.json()

print(f"Got {len(data['results'])} results in {data['response_time']}s")
print(list(data.keys()))

### What comes back

The response is a JSON object. The parts we care about:

- **`results`** — a ranked list; each item has `title`, `url`, `content` (a relevant text snippet), and a `score` (relevance, 0–1).
- **`answer`** — Tavily's own one-line answer (present only because we set `include_answer`).
- **`response_time`**, **`request_id`** — metadata.

`content` is the key field: short, relevant chunks we can feed straight to an LLM without scraping full pages.

In [ ]:
print("Tavily's answer:", data.get("answer"))

print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

## 2 — The same thing with the Python client

Writing the HTTP call by hand is great for understanding, but tedious in practice. The `tavily-python` client wraps exactly that POST request — same parameters, less boilerplate. This is what we'll use for the rest of the notebook.

In [ ]:
from tavily import TavilyClient

# The `client_name` is optional, but it helps track attribution
tavily_client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"], client_name="nv-course-ai-agents"
)

data = tavily_client.search(
    "advancements in AI this week",
    search_depth="advanced",
    max_results=5,
    time_range="week",
)

print(f"{len(data['results'])} results")

In [ ]:
print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

## 3 — A "tool" is just a function

An agent tool is nothing exotic — it's a normal Python function with a clear input and output. Here, we create a tool `internet_search()` that wraps the Tavily Search we saw above. Additionally, we add a small helper that flattens the results into a compact string, which we can then hand back to the model.

In [ ]:
def format_results(results: list[dict]) -> str:
    """Turn search results into a compact, model-friendly string."""
    if not results:
        return "No results found."
    return "\n\n".join(
        f"({i + 1}) Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
        for i, r in enumerate(results)
    )


def internet_search(
    query: str, search_depth: str = "advanced", max_results: int = 5
) -> list[dict]:
    """Search the web and return a list of {title, url, content} results."""
    data = tavily_client.search(
        query, search_depth=search_depth, max_results=max_results
    )

    results = format_results(data["results"])

    return results


print(
    internet_search(
        "Which teams qualified to the Round of 16 in the FIFA World Cup 2026?",
        max_results=3,
    )
)

## 4 — Let the model decide to call the tool

Up until now, we have only worked with the "tool" part of an agent.

Now the LLM. We point the `openai` SDK at Nebius's OpenAI-compatible endpoint, then **describe** our tool to the model as a JSON schema.

The crucial idea: the model never runs the function itself. When it decides a search is needed, it returns a **tool call** — the function name plus the arguments it wants us to run. Executing it is *our* job.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

MODEL = "nvidia/nemotron-3-super-120b-a12b"

# The schema is how the model "sees" our function: name, purpose, and arguments.
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "internet_search",
        "description": "Search the web for current, factual information. Returns results with title, url, and a content snippet.",
        # The `parameters` here match the function's signature that we defined above.
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query."},
                "search_depth": {
                    "type": "string",
                    "enum": ["basic", "fast", "advanced"],
                    "description": "Search depth. Use 'advanced' for more detailed results.",
                },
                "max_results": {
                    "type": "integer",
                    "description": "How many results to return (1-10).",
                },
            },
            "required": ["query"],
        },
    },
}

In [ ]:
from datetime import datetime

messages = [
    {
        "role": "system",
        "content": f"You are a research assistant. Use web search to get up-to-date, factual information before answering. Today is {datetime.now().strftime('%Y-%m-%d')}",
    },
    {"role": "user", "content": "What were the advancements in AI this week?"},
]

response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[SEARCH_TOOL]
)
assistant_msg = response.choices[0].message

print("Content (usually empty when it wants a tool):", repr(assistant_msg.content))
print("Tool calls the model is requesting:")
for tc in assistant_msg.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")

Notice the model didn't answer — it handed us a **request to act**: "please run `internet_search` with these arguments." 

Running it and returning the result is *our* job. Do that, feed the result back, and the model continues. Wrap that back-and-forth in a loop and you have a basic agent.

In [ ]:
import json

# The model asked us to run a tool. Let's fulfil that request by hand, then hand
# the results back so it can finish -- ONE turn of what will become the loop.

# 1) Record the model's tool-call turn in the conversation history.
messages.append(assistant_msg.model_dump(exclude_none=True))

# 2) Run each tool call the model requested, appending each result as a "tool" message.
for tc in assistant_msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"Running internet_search({args})")
    result = internet_search(**args)  # our function already returns a formatted string
    print(f"Result snippet: {result[:100]} ... {result[-100:]}")
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result,
        }
    )

# 3) Ask the model again -- now it can see the search results and can answer.
followup = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[SEARCH_TOOL]
)
final = followup.choices[0].message

if final.content:
    print("\nFinal answer:\n")
    print(final.content)
else:
    print("\nThe model wants to search again:")
    for tc in final.tool_calls or []:
        print(f"  {tc.function.name}({tc.function.arguments})")
    print("...which is exactly why we wrap this in a loop next.")

## 5 — The Pythonic agentic loop

The whole pattern is a `while` loop over a growing list of messages:

1. Ask the model, offering it the tool.
2. If it returns tool calls → run each one, append the results, loop again.
3. If it returns plain text → it's done; that text is the answer.

We add a `max_steps` guard so a confused model can't loop forever. That's the entire idea behind "agents" — everything fancier is an optimization on top of this loop.

In [ ]:
import json
from datetime import datetime

# Map the tool name the model knows to the actual Python function.
TOOLS = {"internet_search": internet_search}

SYSTEM_PROMPT = f"""You are a research assistant. Use the internet_search tool to gather current, factual information before answering. Base your answer only on  what you find, and cite the source URL for every claim. Search more than once if you need different angles. Today is {datetime.now().strftime("%Y-%m-%d")}"""


def run_agent(
    question: str, model: str = MODEL, max_steps: int = 10, verbose: bool = True
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=[SEARCH_TOOL]
        )
        msg = response.choices[0].message
        messages.append(
            msg.model_dump(exclude_none=True)
        )  # keep the assistant turn (incl. tool_calls)

        # No tool calls means the model is done reasoning -> this is the answer.
        if not msg.tool_calls:
            if verbose:
                print(f"[bold green]Step {step}: final answer[/bold green]")
            return msg.content

        # Otherwise, run each requested tool call and feed the result back.
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"[bold cyan]Step {step}: internet_search[/bold cyan] {args}")
            result = TOOLS[tc.function.name](**args)  # already a formatted string
            if verbose:
                print(f"Result snippet: {result[:100]} ... {result[-100:]}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                }
            )

    return "Stopped: reached max_steps without a final answer."

In [ ]:
answer = run_agent(
    "What were the 3 biggest advancements in AI this week?"
)
print()
print(answer)

## 6 — Bonus: how good are different models at tool calling?

Not every model is equally reliable at *deciding* to call a tool, formatting *valid* arguments, and *stopping* when done. Because our loop is model-agnostic, we can point it at several Nebius models and eyeball the difference. Swap in any model IDs from the [Nebius playground](https://tokenfactory.nebius.com); bad names are caught by the `try/except` below.

In [ ]:
CANDIDATE_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b",
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    "MiniMaxAI/MiniMax-M2.5",
    "zai-org/GLM-5.2",
]

question = "What is the latest stable version of Python, and when was it released?"

for model in CANDIDATE_MODELS:
    print(f"[bold]{model}[/bold]")
    try:
        answer = run_agent(question, model=model, max_steps=5, verbose=False)
        print(answer[:400])
    except Exception as e:
        print(f"[red]failed: {type(e).__name__}: {e}[/red]")
    print("-" * 80)

## 7 — Where this breaks, and what's next

Our 20-line agent works, but it's naive:

- **No planning.** It reacts one search at a time; it doesn't decompose a big question into sub-tasks.
- **Redundant searches.** Nothing stops it re-querying the same thing.
- **No source-quality control.** It trusts whatever ranks highest.
- **Context bloat.** Every result is stuffed back into the message list; long runs blow past the context window.

Fixing these is the job of a real **search pipeline** — query planning, result deduplication, source filtering, and summarization — which we build in the next chapter. From there we graduate to a full multi-agent **competitor-research agent** (see the project code in the repo root) that plans, delegates to parallel sub-agents, and fact-checks its own output.